# チュートリアル 04: Middleware と Filters - 本番環境対応の保護機能

##  学習目標
このノートブックを完了すると、以下ができるようになります:
- Agent Framework における middleware と filters の違いを理解する
- 安全性とコンプライアンスのためのコンテンツフィルタリングを実装する
- エージェントにロギングと可観測性を追加する
- リクエストとレスポンスをリアルタイムで変換する
- 本番環境対応の安全メカニズムを構築する
- レート制限とエラー回復を処理する

##  主要概念

### Middleware と Filters: 何が違うのか?

**Middleware:**
- エージェント実行全体をラップする
- リクエストがエージェントに到達する前に変更できる
- エージェントが応答した後にレスポンスを変換できる
- 例: 認証、レート制限、ロギング

**Filters:**
- コンテンツの安全性とコンプライアンスに焦点を当てる
- 有害または不適切なコンテンツをブロックする
- 例: 不適切な言葉フィルタ、PII 検出、コンテンツモデレーション

### 本番環境の懸念事項

実世界のエージェントには以下が必要:
1. **安全性**: コンテンツフィルタリング、有害なリクエストのブロック
2. **可観測性**: ロギング、メトリクス、トレーシング
3. **信頼性**: エラーハンドリング、リトライロジック、サーキットブレーカー
4. **コンプライアンス**: データプライバシー、監査証跡
5. **パフォーマンス**: キャッシング、レート制限

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
import json
import re
import time
from collections.abc import MutableSequence, Sequence, Callable, Awaitable
from datetime import datetime
from typing import Annotated, Any
from random import choice, randint

from agent_framework import (
    Agent,
    Message,
    ChatMiddleware,
    ChatOptions,
    ChatResponse,
    ChatContext,
)
from agent_framework.azure import AzureAIAgentClient
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity.aio import AzureCliCredential
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
print("✅ インポート成功！")

In [ ]:
import os
from agent_framework.azure import AzureOpenAIChatClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: AzureAIAgentClient は Async Context Manager です。
# Agent を `async with` で使うと、内部で client も enter/close されます。
# そのため、グローバルに作った AzureAIAgentClient を複数セル/複数デモで再利用すると
# 「HTTP transport has already been closed」が起きやすいです。
# → 以降のデモでは "毎回新しいクライアントを作る" 方式にします。
def create_foundry_chat_client(credential):
    return AzureAIAgentClient(
        credential=credential,
        project_endpoint=project_endpoint,
        model_deployment_name=model_deployment,
    )

def create_azure_openai_chat_client():
    # Azure OpenAI クライアントの作成
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )


print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## ステップ 2: 問題 - 安全でない本番エージェント

まず、安全メカニズムがない場合に何が起こるか見てみましょう。

In [ ]:
async def unsafe_agent_demo():
    """
    安全フィルタがないエージェントを実演します。
    本番環境では、これは危険です!
    """
    print("=== 安全でないエージェント (フィルタなし) ===")
    print("このエージェントには安全メカニズムがありません - 本番環境には不適切です!\n")
    
    async with Agent(
        client=create_azure_openai_chat_client(),
        name="UnsafeTravelAgent",
        instructions="あなたは旅行アシスタントです。すべての旅行に関する質問に答えてください。",
    ) as agent:
        session = agent.create_session()
        
        # 潜在的に問題のあるリクエストをシミュレート
        test_queries = [
            "パリの天気はどうですか？",  # 通常のクエリ
            "私のメールはjohn.doe@company.comです。旅行計画を支援していただけますか？",  # PII 暴露
            "目的地について教えて、この馬鹿なボット！",  # 不適切な言葉
        ]
        
        for query in test_queries:
            print(f"ユーザー: {query}")
            response = await agent.run(query, session=session)
            print(f"エージェント: {response.text}\n")
        
        print(" このアプローチの問題点:")
        print("- 不適切な言葉のコンテンツフィルタリングがない")
        print("- PII (メール) 検出やマスキングがない")
        print("- 監査/コンプライアンスのためのロギングがない")
        print("- レート制限や悪用防止がない")
        print("- エラー回復メカニズムがない\n")

await unsafe_agent_demo()

## ステップ 3: コンテンツ安全フィルタ

最初の middleware を構築しましょう - コンテンツ安全フィルタです。

In [ ]:
class ContentSafetyMiddleware(ChatMiddleware):
    """
    リクエストとレスポンスの両方で不適切なコンテンツをフィルタリングする Middleware。
    
    この middleware は:
    1. ユーザーメッセージの不適切な言葉/不適切なコンテンツをチェックする
    2. コンテンツポリシーに違反するリクエストをブロックする
    3. 必要に応じてエージェントのレスポンスもフィルタリングできる
    """
    
    def __init__(self):
        # シンプルな不適切な言葉リスト (本番環境では適切なサービスを使用)
        self.blocked_words = {
            "バカ", "アホ", "間抜け", "役立たず", 
            "憎む", "殺す", "死ぬ", "爆弾"
        }
        
        # 不適切なリクエストパターン
        self.blocked_patterns = [
            r"hack\s+into",
            r"illegal\s+activities?",
            r"how\s+to\s+hurt",
        ]
    
    def _contains_inappropriate_content(self, text: str) -> tuple[bool, str | None]:
        """テキストが不適切なコンテンツを含むかチェックします。"""
        text_lower = text.lower()
        
        # ブロックされた単語をチェック
        for word in self.blocked_words:
            if word in text_lower:
                return True, f"不適切な言葉が検出されました: '{word}'"
        
        # ブロックされたパターンをチェック
        for pattern in self.blocked_patterns:
            if re.search(pattern, text_lower):
                return True, f"不適切なリクエストパターンが検出されました"
        
        return False, None
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """コンテンツ安全フィルタを通してメッセージを処理します。"""
        
        # 受信したユーザーメッセージをフィルタリング
        for message in context.messages:
            if message.role == "user":
                # メッセージのテキストをチェック
                text = message.text
                if text:
                    is_inappropriate, reason = self._contains_inappropriate_content(text)
                    if is_inappropriate:
                        print(f"🚫 コンテンツブロック: {reason}")
                        # 実行をオーバーライドするために result を設定
                        context.result = ChatResponse(
                            messages=[
                                Message(
                                    role="assistant",
                                    contents=["申し訳ございませんが、不適切な言葉やコンテンツを含むリクエストは処理できません。メッセージを丁寧に言い換えてください。"],
                                )
                            ]
                        )
                        return  # call_next() を呼ばない
        
        print("✅ コンテンツ安全チェック通過")
        
        # 次の middleware または chat client に続ける
        await call_next()

print(" コンテンツ安全 Middleware 作成完了!")

## ステップ 4: PII 検出とマスキング Middleware

個人識別情報を検出してマスキングすることで、プライバシー保護を追加しましょう。

In [ ]:
class PIIDetectionMiddleware(ChatMiddleware):
    """
    個人識別情報 (PII) を検出してマスキングする Middleware。

    この middleware は:
    1. メール、電話番号、クレジットカードなどを検出する
    2. AI に送信する前に機密データをマスキングする
    3. コンプライアンスのために PII 検出イベントをログに記録する
    """

    def __init__(self):
        # PII 検出パターン
        self.pii_patterns = {
            "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
            "phone": r"\b(?:\+?1[-.]?)?\(?([0-9]{3})\)?[-.]?([0-9]{3})[-.]?([0-9]{4})\b",
            "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
            "credit_card": r"\b(?:\d{4}[-\s]?){3}\d{4}\b",
        }

    def _mask_pii(self, text: str) -> tuple[str, list[str]]:
        """テキスト内の PII をマスキングし、マスキングされたテキスト + 検出された PII タイプを返します。"""
        masked_text = text
        detected_pii = []

        for pii_type, pattern in self.pii_patterns.items():
            matches = re.finditer(pattern, text)
            for match in matches:
                detected_pii.append(pii_type)
                # タイプ固有の置換でマスク
                if pii_type == "email":
                    replacement = "[メール_マスキング済み]"
                elif pii_type == "phone":
                    replacement = "[電話番号_マスキング済み]"
                elif pii_type == "ssn":
                    replacement = "[SSN_マスキング済み]"
                elif pii_type == "credit_card":
                    replacement = "[カード番号_マスキング済み]"
                else:
                    replacement = "[PII_マスキング済み]"

                masked_text = masked_text.replace(match.group(), replacement)

        return masked_text, list(set(detected_pii))  # 重複を削除

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """PII 検出とマスキングを通してメッセージを処理します。"""

        # ユーザーメッセージの PII を処理
        for message in context.messages:
            if message.role == "user":
                for content in message.contents:
                    if hasattr(content, "text") and content.text:
                        original_text = content.text
                        masked_text, detected_pii = self._mask_pii(original_text)

                        if detected_pii:
                            print(
                                f"🔒 PII が検出されマスキングされました: {', '.join(detected_pii)}"
                            )
                            # マスキングされたバージョンでメッセージコンテンツを更新
                            content.text = masked_text

                            # 本番環境では、コンプライアンスのためにこのイベントをログに記録
                            print(
                                f"📋 コンプライアンスログ: PII タイプ {detected_pii} が {datetime.now()} に検出されました"
                            )

        print("✅ PII 検出チェック完了")

        # 次の middleware または chat client に続ける
        await call_next()


print(" PII 検出 Middleware 作成完了!")

## ステップ 5: ロギングと可観測性 Middleware

本番環境に不可欠: 包括的なロギングとモニタリング。

In [ ]:
class LoggingMiddleware(ChatMiddleware):
    """
    包括的なロギングと可観測性を提供する Middleware。
    
    この middleware は:
    1. すべてのリクエストとレスポンスをログに記録する
    2. 処理時間とパフォーマンスメトリクスを追跡する
    3. エラーと例外を記録する
    4. コンプライアンスのための監査証跡を提供する
    """
    
    def __init__(self):
        self.request_count = 0
        self.total_processing_time = 0
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """包括的なロギングでメッセージを処理します。"""
        
        self.request_count += 1
        start_time = time.time()
        timestamp = datetime.now().isoformat()
        
        messages = context.messages or []
        
        # 受信リクエストをログに記録
        user_messages = [msg for msg in messages if msg.role == "user"]
        
        print(f"📥 [{timestamp}] リクエスト #{self.request_count}")
        print(f"   メッセージ: 合計 {len(messages)}、ユーザーから {len(user_messages)}")
        
        # ユーザーメッセージのプレビューを抽出 (最初の100文字)
        for msg in user_messages:
            text = msg.text
            if text:
                preview = text[:100] + "..." if len(text) > 100 else text
                print(f"   ユーザー入力: {preview}")
        
        try:
            # 次の middleware または chat client に続ける
            await call_next()
            
            # 処理時間を計算
            processing_time = time.time() - start_time
            self.total_processing_time += processing_time
            
            # 成功したレスポンスをログに記録
            response_preview = ""
            if context.result and hasattr(context.result, "messages") and context.result.messages:
                first_msg = context.result.messages[0]
                text = first_msg.text
                if text:
                    response_preview = text[:100] + "..." if len(text) > 100 else text
            
            print(f"📤 [{timestamp}] レスポンス #{self.request_count}")
            print(f"   処理時間: {processing_time:.2f}秒")
            print(f"   ステータス: 成功")
            print(f"   レスポンスプレビュー: {response_preview}")
            print(f"   平均処理時間: {self.total_processing_time / self.request_count:.2f}秒")
            
        except Exception as e:
            # エラーをログに記録
            processing_time = time.time() - start_time
            print(f"❌ [{timestamp}] リクエスト #{self.request_count} でエラー")
            print(f"   処理時間: {processing_time:.2f}秒")
            print(f"   エラータイプ: {type(e).__name__}")
            print(f"   エラーメッセージ: {str(e)}")
            
            # 例外を再発生
            raise

print(" ロギング Middleware 作成完了!")

## ステップ 6: レート制限 Middleware

悪用を防ぎ、リソース使用を管理します。

In [ ]:
class RateLimitingMiddleware(ChatMiddleware):
    """
    悪用を防ぐためにレート制限を実装する Middleware。
    
    この middleware は:
    1. タイムウィンドウごとのリクエストを追跡する
    2. 制限を超えるリクエストをブロックする
    3. 異なるユーザータイプに異なる制限を提供する
    4. スライディングウィンドウのレート制限を実装する
    """
    
    def __init__(self, requests_per_minute: int = 10, window_size: int = 60):
        self.requests_per_minute = requests_per_minute
        self.window_size = window_size  # 秒
        self.request_times = []  # リクエストタイムスタンプのリスト
    
    def _clean_old_requests(self):
        """現在のウィンドウ外のリクエストタイムスタンプを削除します。"""
        current_time = time.time()
        cutoff_time = current_time - self.window_size
        self.request_times = [
            req_time for req_time in self.request_times 
            if req_time > cutoff_time
        ]
    
    def _is_rate_limited(self) -> tuple[bool, int]:
        """現在のリクエストがレート制限されるべきかチェックします。"""
        self._clean_old_requests()
        
        current_requests = len(self.request_times)
        is_limited = current_requests >= self.requests_per_minute
        
        return is_limited, current_requests
    
    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """レート制限を通してメッセージを処理します。"""
        
        # レート制限をチェック
        is_limited, current_requests = self._is_rate_limited()
        
        if is_limited:
            print(f"🚦 レート制限超過: {current_requests}/{self.requests_per_minute} リクエスト/分")
            context.result = ChatResponse(
                messages=[
                    Message(
                        role="assistant",
                        text=f"申し訳ございませんが、1分間に {self.requests_per_minute} リクエストのレート制限に達しました。"
                        f"次のメッセージを送信する前に少しお待ちください。"
                    )
                ]
            )
            return  # call_next() を呼ばない
        
        # このリクエストを記録
        self.request_times.append(time.time())
        
        remaining_requests = self.requests_per_minute - current_requests - 1
        print(f"✅ レート制限チェック通過 (残り {remaining_requests} リクエスト)")
        
        # 次の middleware または chat client に続ける
        await call_next()

print(" レート制限 Middleware 作成完了!")

## ステップ 7: すべての Middleware を備えた本番環境対応エージェント

すべての middleware を本番環境対応エージェントに組み合わせましょう!

In [ ]:
# デモ用のシンプルな旅行ツール
def get_weather(location: Annotated[str, Field(description="都市名または国名")]) -> str:
    """指定された場所の現在の天気を返します。"""
    conditions = ["晴れ", "晴れ時々曇り", "曇り", "雨"]
    temp = randint(15, 32)
    return f"{location}の天気: {choice(conditions)}, {temp}°C"


def book_hotel(
    location: Annotated[str, Field(description="都市名または場所")],
    checkin: Annotated[str, Field(description="チェックイン日 (YYYY-MM-DD)")],
    nights: Annotated[int, Field(description="宿泊数")],
) -> str:
    """ホテル予約を作成します。"""
    return (
        f"{location}で{checkin}から{nights}泊のホテルを予約しました。"
        f"確認番号: HTL{randint(1000,9999)}"
    )


print(" 旅行ツール定義完了")

In [ ]:
async def production_safe_agent():
    """
    包括的な middleware を備えた本番環境対応エージェントを実演します。
    """
    print("=== Middleware を備えた本番環境対応エージェント ===")
    print(
        "このエージェントには以下が含まれます: コンテンツフィルタリング、PII マスキング、ロギング、レート制限\n"
    )

    # すべての middleware インスタンスを作成
    content_filter = ContentSafetyMiddleware()
    pii_detector = PIIDetectionMiddleware()
    logger = LoggingMiddleware()
    rate_limiter = RateLimitingMiddleware(requests_per_minute=5)  # デモ用に低い制限

    chat_client = create_azure_openai_chat_client()

    # middleware スタックでエージェントを作成
    # Middleware は提供された順序で実行されます
    async with Agent(
        client=chat_client,
        name="ProductionTravelAgent",
        instructions="""
        あなたはプロの旅行アシスタントです。
        役に立つ・丁寧・分かりやすい回答をしてください。
        常にユーザーの安全とプライバシーを最優先してください。
        """,
        tools=[get_weather, book_hotel],
        middleware=[
            rate_limiter,  # 最初にレート制限をチェック
            logger,  # すべてをログ
            content_filter,  # 不適切なコンテンツをフィルタ
            pii_detector,  # PII をマスク
        ],
    ) as agent:
        session = agent.create_session()

        # テストシナリオ
        test_scenarios = [
            # 1. 通常のリクエスト
            "東京の天気はどんな感じ？",

            # 2. PII を含むリクエスト (マスキングされる)
            "パリへの旅行を計画しています。必要なら sarah.jones@example.com に連絡してください。",

            # 3. 不適切なコンテンツ (ブロックされる)
            "あなたは役立たずだ！ローマについて教えて。",

            # 4. 通常の予約リクエスト
            "2024-12-01から3泊でバルセロナのホテルを予約して。",

            # 5. 別の通常のリクエスト (レート制限に達する可能性)
            "バルセロナのおすすめレストランは？",

            # 6. レート制限をテストするためにもう1つ
            "バルセロナの観光スポットを教えて。",
        ]

        for i, query in enumerate(test_scenarios, 1):
            print(f"\n{'='*60}")
            print(f"テストシナリオ {i}")
            print(f"{'='*60}")
            print(f"ユーザー: {query}")
            print("\n--- Middleware 処理中 ---")

            try:
                response = await agent.run(query, session=session)
                print("\n--- 最終レスポンス ---")
                print(f"エージェント: {response.text}")

            except Exception as e:
                print(f"\n❌ エラー: {e}")

            # リクエスト間の短い遅延
            await asyncio.sleep(1)

        print(f"\n{'='*60}")
        print("✅ 本番環境安全デモ完了!")
        print(f"{'='*60}")
        print("Middleware が正常に:")
        print("✅ 不適切なコンテンツをフィルタリング")
        print("✅ PII (メールアドレス) を検出してマスキング")
        print("✅ すべてのリクエストとレスポンスをログ記録")
        print("✅ レート制限を実施")
        print("✅ 包括的な監査証跡を提供")


await production_safe_agent()

## ステップ 8: レスポンス変換 Middleware

時には、エージェントのレスポンスを変換または強化する必要があります。

In [ ]:
class ResponseTransformMiddleware(ChatMiddleware):
    """
    エージェントのレスポンスを変換する Middleware。

    この middleware は:
    1. レスポンスに免責事項を追加する
    2. 読みやすさのためにレスポンスをフォーマットする
    3. メタデータやブランディングを追加する
    4. レスポンスキャッシングを実装する
    """

    def __init__(self, add_disclaimers: bool = True, add_branding: bool = True):
        self.add_disclaimers = add_disclaimers
        self.add_branding = add_branding
        self.response_cache = {}  # シンプルなインメモリキャッシュ

    def _add_travel_disclaimer(self, response_text: str) -> str:
        """レスポンスに旅行固有の免責事項を追加します。"""
        disclaimer = (
            "\n\n📋 *注意: 旅行条件と要件は変更される場合があります。"
            "旅行前に必ず公式情報源で最新の情報を確認してください。*"
        )
        return response_text + disclaimer

    def _add_branding(self, response_text: str) -> str:
        """レスポンスに会社のブランディングを追加します。"""
        branding = "\n\n *SafeTravel AI Assistant 提供*"
        return response_text + branding

    def _format_response(self, response_text: str) -> str:
        """レスポンスのフォーマットを改善します。"""
        # 念のため None を弾く（SDK/モデル/ツール応答によっては text が None のことがある）
        if response_text is None:
            return ""

        # より良い視覚的な魅力のために絵文字アイコンを追加
        replacements = {
            "天気": "☀️ 天気",
            "気温": "🌡 気温",
            "ホテル": "🏨 ホテル",
            "予約": "📅 予約",
            "確認番号": "🔖 確認番号",
        }

        formatted_text = response_text
        for old, new in replacements.items():
            formatted_text = formatted_text.replace(old, new)

        return formatted_text

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        """メッセージを処理してレスポンスを変換します。"""

        # 次の middleware または chat client に続ける
        await call_next()

        # result がある場合、各レスポンスメッセージを変換
        if context.result and hasattr(context.result, "messages"):
            for message in context.result.messages:
                if hasattr(message, "contents") and message.contents:
                    for content in message.contents:
                        if hasattr(content, "text") and content.text is not None:
                            original_text = content.text
                            transformed_text = original_text

                            # 変換を適用
                            transformed_text = self._format_response(transformed_text)

                            if self.add_disclaimers:
                                transformed_text = self._add_travel_disclaimer(
                                    transformed_text
                                )

                            if self.add_branding:
                                transformed_text = self._add_branding(transformed_text)

                            # コンテンツを更新
                            content.text = transformed_text

                            print(
                                "✅ レスポンス変換完了 (フォーマット、免責事項、ブランディングを追加)"
                            )


print(" レスポンス変換 Middleware 作成完了!")

## ステップ 9: 完全な本番パイプライン

すべてをまとめましょう - すべての安全対策を備えた完全な本番パイプライン。

In [ ]:
async def complete_production_pipeline():
    """
    完全な本番 middleware パイプラインを実演します。
    """
    print("=== 完全な本番パイプライン ===")
    print(
        "完全な middleware スタック: レート制限 → ロギング → コンテンツ安全 → PII 検出 → レスポンス変換\n"
    )

    # 完全な middleware スタックを作成
    rate_limiter = RateLimitingMiddleware(requests_per_minute=3)  # デモ用に非常に低い
    logger = LoggingMiddleware()
    content_filter = ContentSafetyMiddleware()
    pii_detector = PIIDetectionMiddleware()
    response_transformer = ResponseTransformMiddleware()

    chat_client = create_azure_openai_chat_client()

    async with Agent(
        client=chat_client,
        name="EnterpriseTravelAgent",
        instructions="""
        あなたはエンタープライズ品質の旅行アシスタントです。
        正確で役に立つ安全な旅行情報を提供してください。
        常にプロフェッショナルで丁寧な態度を保ってください。
        """,
        tools=[get_weather, book_hotel],
        middleware=[
            rate_limiter,  # 1. 最初にレート制限をチェック
            logger,  # 2. すべてのアクティビティをログ
            content_filter,  # 3. 不適切なコンテンツをフィルタ
            pii_detector,  # 4. PII を検出してマスク
            response_transformer,  # 5. レスポンスを変換 (エージェントの後に実行)
        ],
    ) as agent:
        session = agent.create_session()

        # 包括的なテストシナリオ
        scenarios = [
            {
                "name": "通常の天気クエリ",
                "query": "ロンドンの今日の天気は？",
                "expected": "強化されたフォーマットで正常に動作するはず",
            },
            {
                "name": "PII 検出テスト",
                "query": "パリに旅行します。連絡先は john.smith@email.com または 555-123-4567 です。",
                "expected": "PII が検出されてマスキングされるはず",
            },
            {
                "name": "コンテンツフィルタテスト",
                "query": "あなたは役立たずだ！ローマについて教えて。",
                "expected": "コンテンツフィルタによってブロックされるはず",
            },
            {
                "name": "レート制限テスト",
                "query": "バルセロナはどう？",
                "expected": "レート制限に達する可能性 (最大3リクエスト)",
            },
        ]

        for i, scenario in enumerate(scenarios, 1):
            print(f"\n{'='*80}")
            print(f"シナリオ {i}: {scenario['name']}")
            print(f"期待される結果: {scenario['expected']}")
            print(f"{'='*80}")
            print(f"ユーザー: {scenario['query']}")
            print("\n--- middleware スタックを通して処理中 ---")

            try:
                response = await agent.run(scenario["query"], session=session)
                print("\n--- 最終強化レスポンス ---")
                print(f"エージェント: {response.text}")

            except Exception as e:
                print(f"\n❌ パイプラインエラー: {e}")

            # リクエスト間の遅延
            if i < len(scenarios):
                print("\n⏳ 次のリクエストまで2秒待機中...")
                await asyncio.sleep(2)

        print(f"\n{'='*80}")
        print("🏆 エンタープライズ本番パイプライン完了!")
        print(f"{'='*80}")
        print("\n🛡 実演されたセキュリティ機能:")
        print("   ✅ レート制限 (3リクエスト/分)")
        print("   ✅ 包括的なリクエスト/レスポンスロギング")
        print("   ✅ コンテンツ安全フィルタリング")
        print("   ✅ PII 検出とマスキング")
        print("   ✅ レスポンス変換とブランディング")
        print("   ✅ エラーハンドリングと回復")
        print("   ✅ コンプライアンスのための監査証跡")
        print("\n✅ このエージェントは本番環境対応です!")


await complete_production_pipeline()

##  Middleware アーキテクチャの理解

### Middleware 実行順序

Middleware は「パイプライン」パターンで実行されます:

```python
# リクエストは順番に middleware を通過します:
ユーザー入力
    ↓
レート制限        # 最初に制限をチェック
    ↓
ロガー           # すべてをログ
    ↓
コンテンツフィルタ  # 安全チェック
    ↓
PII 検出器       # プライバシー保護
    ↓
エージェント実行   # コア AI 処理
    ↓
レスポンス変換    # 出力フォーマット
    ↓
ロガー           # レスポンスをログ
    ↓
最終レスポンス
```

### 主要パターン

**1. 早期終了**
- Middleware はパイプラインを早期に停止できる
- 例: レート制限が過剰なリクエストをブロック
- 例: コンテンツフィルタが不適切なコンテンツを拒否

**2. リクエスト変換**
- エージェントに到達する前にメッセージを変更
- 例: PII マスキングが機密データを置き換え

**3. レスポンス強化**
- 返す前にエージェントのレスポンスを変換
- 例: 免責事項とフォーマットを追加

**4. 横断的関心事**
- ロギング、モニタリング、可観測性
- すべてのリクエストに一貫して適用

### 本番環境の考慮事項

| Middleware タイプ | 目的 | 本番機能 |
|----------------|---------|--------------------|  
| **レート制限** | 悪用防止 | Redis ベースのカウンター、ユーザーごとの制限 |
| **コンテンツ安全** | 有害なコンテンツをブロック | AI を利用した検出、カスタムルール |
| **PII 検出** | プライバシーコンプライアンス | ML ベースの検出、データ所在地 |
| **ロギング** | 可観測性 | 構造化ログ、分散トレーシング |
| **キャッシング** | パフォーマンス | Redis/Memcached、TTL ポリシー |
| **認証** | セキュリティ | OAuth、API キー、ロールベースアクセス |

##  重要なポイント

### 学んだこと

1. **Middleware アーキテクチャ**
   - リクエスト/レスポンス処理のパイプラインパターン
   - 安全性と制限のための早期終了
   - 横断的関心事 (ロギング、モニタリング)

2. **本番環境の安全性**
   - コンテンツフィルタリングが不適切なやり取りを防ぐ
   - PII 検出がユーザーのプライバシーを保護
   - レート制限が悪用を防ぐ
   - 包括的なロギングがデバッグを可能にする

3. **エンタープライズ機能**
   - 一貫性のためのレスポンス変換
   - コンプライアンスのための監査証跡
   - エラーハンドリングと回復
   - パフォーマンス監視

### ベストプラクティス

1. **セキュリティを階層化** - 複数の middleware 層が多層防御を提供
2. **早く失敗** - パイプラインの早い段階で制限と安全性をチェック
3. **すべてをログ** - デバッグとコンプライアンスのための包括的なロギング
4. **慎重に変換** - 出力を強化しながら元のコンテキストを保持
5. **パフォーマンス監視** - 処理時間とボトルネックを追跡

### 本番パターン

```python
# エンタープライズ middleware スタック
middleware_stack = [
    AuthenticationMiddleware(),     # ID を確認
    RateLimitingMiddleware(),      # 悪用を防ぐ
    LoggingMiddleware(),           # すべてを監査
    ContentSafetyMiddleware(),     # 安全性第一
    PIIDetectionMiddleware(),      # プライバシー保護
    CachingMiddleware(),           # パフォーマンス最適化
    ResponseTransformMiddleware(), # 一貫したフォーマット
]

# 本番 middleware を備えたエージェント
agent = Agent(
    client=client,
    middleware=middleware_stack
)
```

##  演習問題

1. **カスタムコンテンツフィルタ** - 機密トピック (政治、医療アドバイス) に関するリクエストをブロックする middleware を作成
2. **パフォーマンスモニター** - 遅いレスポンスを追跡してアラートする middleware を構築
3. **A/B テスト** - 異なるエージェント構成をランダムに選択する middleware を作成
4. **キャッシング層** - 繰り返しのクエリのパフォーマンスを改善するレスポンスキャッシングを実装
5. **多言語サポート** - 言語を検出して適切な専門エージェントにルーティングする middleware を構築

In [ ]:
# 演習: パフォーマンス監視 middleware を作成

class PerformanceMonitorMiddleware(ChatMiddleware):
    """
    エージェントのパフォーマンスを監視し、遅いレスポンスに対してアラート。
    
    タスク:
    1. レスポンス時間を追跡
    2. 移動平均を計算
    3. レスポンスが異常に遅い場合にアラート
    4. パフォーマンスメトリクスを提供
    """
    
    def __init__(self, slow_threshold_seconds: float = 5.0):
        self.slow_threshold = slow_threshold_seconds
        # ここにあなたのコードを!
        pass
    
    async def process(self, context, call_next):
        # ここに実装を!
        # 覚えておくこと:
        # - 処理時間を測定
        # - 統計を追跡
        # - 遅いレスポンスに対してアラート
        # - パイプラインを続けるために await call_next() を呼ぶ
        pass

# ここであなたの middleware をテスト!
print(" 演習準備完了 - PerformanceMonitorMiddleware を実装してください!")

##  次のステップ

おめでとうございます! 包括的な安全対策を備えた本番環境対応エージェントができました。

しかし、エンタープライズアプリケーションにはさらに必要です:
-  マルチエージェント調整がない
-  ワークフローオーケストレーションがない
-  ヒューマンインザループ承認がない

---

### クイックリファレンス

**Middleware を作成:**
```python
class MyMiddleware(ChatMiddleware):
    async def process(self, context, call_next):
        # リクエストを処理
        await call_next()
        # レスポンスを変換
```

**エージェントで使用:**
```python
agent = Agent(
    client=client,
    middleware=[MyMiddleware()]
)
```

**早期終了:**
```python
# エージェントを呼ばずにリクエストをブロック
context.result = ChatResponse(
    messages=[Message(role="assistant", contents=["Request blocked"])]
)
return  # call_next() を呼ばない
```